# SYMCA Attendance Prediction — 01 EDA (Final)

**Project:** Classroom Attendance Prediction Using Academic Schedule and Historical Attendance Data

**Purpose:** Explore and validate the original attendance dataset before feature engineering and model training.

This notebook is intentionally limited to **EDA and data validation**. It does not create model features and it does not modify the raw CSV on disk.

The DSML brief requires the project to use original historical attendance data, convert it to CSV, perform cleaning/feature engineering, and maintain separate notebooks for EDA, feature engineering, and model training.


## 1. Import Libraries

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("Libraries imported successfully.")


## 2. Locate the Raw CSV in Kaggle

Run this cell first. It prints the exact path of every file available under `/kaggle/input`.

**Important:** Keep `symca_raw.csv` as the original/raw dataset. Do not overwrite it during EDA.


In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))


## 3. Load the Raw Dataset

After running the previous cell, replace `DATA_PATH` with the exact Kaggle path shown for `symca_raw.csv`.


In [ ]:
# Example:
# DATA_PATH = "/kaggle/input/symca-attendance-prediction/symca_raw.csv"

DATA_PATH = "/kaggle/input/YOUR-DATASET-NAME/symca_raw.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape (rows, columns):", df.shape)
display(df.head())


## 4. Dataset Structure

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))


In [ ]:
print("Dataset information:")
df.info()

print("\nSummary statistics:")
display(df.describe(include="all").T)


## 5. Missing Values and Duplicate Records

Missing values and duplicate records are checked before any cleaning.

A missing `Faculty_ID` is not automatically replaced in the raw data. The handling decision belongs in the Feature Engineering/Cleaning notebook.


In [ ]:
missing = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2)
})

display(missing[missing["Missing Values"] > 0])

print("Total missing cells:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))


## 6. Date Validation

In [ ]:
eda = df.copy()

eda["Date"] = pd.to_datetime(
    eda["Date"],
    errors="coerce"
)

print("Minimum date:", eda["Date"].min().date())
print("Maximum date:", eda["Date"].max().date())
print("Invalid dates:", int(eda["Date"].isna().sum()))


## 7. Attendance Percentage Validation

Attendance percentage should follow:

**Attendance Percentage = (Students Present / Total Enrolled Students) × 100**

The calculated value is used only for validation here. The raw target column is not overwritten.


In [ ]:
calculated_attendance = (
    eda["Students Present"] /
    eda["Total Enrolled Students"]
) * 100

difference = (
    calculated_attendance -
    eda["Attendence Percentage"]
).abs()

print("Maximum difference:", difference.max())
print("Mean difference:", difference.mean())

print("\nRows with a difference greater than 0.01 percentage point:",
      int((difference > 0.01).sum()))

display(
    eda.loc[
        difference > 0.01,
        [
            "Date",
            "Subject",
            "Total Enrolled Students",
            "Students Present",
            "Attendence Percentage"
        ]
    ].head(10)
)


## 8. Day-of-Week Validation

In [ ]:
calculated_day = eda["Date"].dt.day_name()

if "Day of Week" in eda.columns:
    mismatch = calculated_day != eda["Day of Week"]
    print("Incorrect Day of Week values:", int(mismatch.sum()))

    if mismatch.any():
        display(
            eda.loc[mismatch, ["Date", "Day of Week"]].head(10)
        )
else:
    print("Day of Week column is not present.")


## 9. Time Format Quality Check

The raw dataset contains start/end times as text. This cell checks whether they follow the expected `HH.MM AM/PM` pattern.

Time normalization will be performed later in the Feature Engineering notebook.


In [ ]:
time_pattern = re.compile(r"^\d{2}\.\d{2} [AP]M$")

for col in ["Start_Time", "End_Time"]:
    values = eda[col].astype(str)
    valid = values.str.match(time_pattern, na=False)

    print(f"{col}:")
    print("  Valid format:", int(valid.sum()))
    print("  Unusual format:", int((~valid).sum()))

    if (~valid).any():
        display(
            eda.loc[~valid, [col]].drop_duplicates().head(20)
        )


## 10. Attendance Target Statistics

In [ ]:
target = "Attendence Percentage"

print("Attendance statistics:")
display(eda[target].describe().to_frame("Attendance %"))

print("Average attendance:", round(eda[target].mean(), 2), "%")
print("Minimum attendance:", round(eda[target].min(), 2), "%")
print("Maximum attendance:", round(eda[target].max(), 2), "%")


## 11. Attendance Distribution

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(eda[target].dropna(), bins=20)
plt.xlabel("Attendance Percentage")
plt.ylabel("Number of Lectures")
plt.title("Distribution of Attendance Percentage")
plt.tight_layout()
plt.show()


## 12. Average Attendance by Subject

In [ ]:
subject_avg = (
    eda.groupby("Subject")[target]
       .mean()
       .sort_values()
)

display(subject_avg.to_frame("Average Attendance %"))

plt.figure(figsize=(10, 8))
subject_avg.plot(kind="barh")
plt.xlabel("Average Attendance (%)")
plt.ylabel("Subject")
plt.title("Average Attendance by Subject")
plt.tight_layout()
plt.show()


## 13. Average Attendance by Day of Week

In [ ]:
day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday"
]

day_avg = (
    eda.groupby("Day of Week")[target]
       .mean()
       .reindex(day_order)
)

display(day_avg.to_frame("Average Attendance %"))

plt.figure(figsize=(9, 5))
day_avg.plot(kind="bar")
plt.xlabel("Day of Week")
plt.ylabel("Average Attendance (%)")
plt.title("Average Attendance by Day of Week")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 14. Internal Test Week Analysis

In [ ]:
test_avg = (
    eda.groupby("Internal Test Week")[target]
       .mean()
)

display(test_avg.to_frame("Average Attendance %"))

plt.figure(figsize=(6, 4))
test_avg.plot(kind="bar")
plt.xlabel("Internal Test Week")
plt.ylabel("Average Attendance (%)")
plt.title("Attendance: Internal Test Week vs Normal Week")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 15. Weather Analysis

In [ ]:
weather_avg = (
    eda.groupby("Weather")[target]
       .mean()
       .sort_values()
)

display(weather_avg.to_frame("Average Attendance %"))

plt.figure(figsize=(8, 5))
weather_avg.plot(kind="bar")
plt.xlabel("Weather")
plt.ylabel("Average Attendance (%)")
plt.title("Average Attendance by Weather")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 16. Special Event Analysis

In [ ]:
special_avg = (
    eda.groupby("Special Event")[target]
       .mean()
)

display(special_avg.to_frame("Average Attendance %"))

plt.figure(figsize=(6, 4))
special_avg.plot(kind="bar")
plt.xlabel("Special Event")
plt.ylabel("Average Attendance (%)")
plt.title("Attendance: Special Event vs Normal Lecture")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 17. Practical vs Theory Analysis

In [ ]:
format_avg = (
    eda.groupby("Practical/ Theory")[target]
       .mean()
)

display(format_avg.to_frame("Average Attendance %"))

plt.figure(figsize=(7, 4))
format_avg.plot(kind="bar")
plt.xlabel("Session Type")
plt.ylabel("Average Attendance (%)")
plt.title("Average Attendance: Practical vs Theory")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 18. Attendance Trend Over Time

In [ ]:
daily_avg = (
    eda.groupby("Date")[target]
       .mean()
       .sort_index()
)

plt.figure(figsize=(12, 5))
daily_avg.plot()
plt.xlabel("Date")
plt.ylabel("Average Attendance (%)")
plt.title("Average Attendance Over Time")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


## 19. Numerical Relationships

This is an **EDA-only** correlation view. It must not be used by itself to decide the final model features.

In particular, `Students Present` is mathematically related to the target attendance percentage, so it must not be used as a prediction input when predicting attendance percentage before a lecture.


In [ ]:
numeric_cols = eda.select_dtypes(include=np.number).columns

corr = eda[numeric_cols].corr()

display(
    corr[target]
    .sort_values(ascending=False)
    .to_frame("Correlation with Attendance %")
)


In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation Heatmap — EDA Only")
plt.tight_layout()
plt.show()


## 20. Outlier Check for Attendance Percentage

In [ ]:
q1 = eda[target].quantile(0.25)
q3 = eda[target].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = eda[
    (eda[target] < lower_bound) |
    (eda[target] > upper_bound)
]

print("Q1:", round(q1, 2))
print("Q3:", round(q3, 2))
print("IQR:", round(iqr, 2))
print("Lower bound:", round(lower_bound, 2))
print("Upper bound:", round(upper_bound, 2))
print("Potential outlier rows:", len(outliers))


## 21. Dataset Coverage Summary

In [ ]:
summary = {
    "Rows": len(eda),
    "Columns": len(eda.columns),
    "Date Start": eda["Date"].min().date(),
    "Date End": eda["Date"].max().date(),
    "Subjects": eda["Subject"].nunique(),
    "Sections": eda["Section"].nunique(),
    "Faculty IDs": eda["Faculty_ID"].nunique(dropna=True),
    "Classrooms": eda["Classroom"].nunique(),
    "Missing Cells": int(eda.isna().sum().sum()),
    "Duplicate Rows": int(eda.duplicated().sum())
}

display(pd.DataFrame([summary]))


## 22. Final EDA Findings

For the current raw dataset, the validated observations are:

- The dataset contains **518 lecture records and 23 columns**.
- The observation period is **15 June 2026 to 21 August 2026**.
- The target is **Attendance Percentage**.
- Attendance ranges from **7.84% to 100%**, with an overall mean of approximately **78.38%**.
- There are **no duplicate rows**.
- `Faculty_ID` contains missing values; this will be handled in the cleaning/feature-engineering stage.
- The attendance percentage agrees with `(Students Present / Total Enrolled Students) × 100` apart from small rounding differences.
- `Students Present` must not be used as an input feature for predicting future attendance percentage because it is used to calculate the target.
- `Previous Lecture Attendence` should be verified/recreated chronologically in the Feature Engineering notebook rather than blindly trusted.
- `Assignment Due` currently has no variation in the supplied data, so it cannot provide useful predictive variation unless future/original observations contain both categories.
- Time normalization, historical rolling features, monthly historical averages, day-of-semester, holiday gaps, and examination-week flags belong in Notebook 02.


## 23. EDA Conclusion

The raw dataset is suitable to proceed to the **Feature Engineering** stage.

The next notebook must preserve chronological order and create historical features using only information available before the lecture being predicted. This prevents target leakage and makes the subsequent regression experiment more realistic.
